# Purpose :

Predict the question a)

## Ideas :

- Learn to predict foreach team of the pool the result of a 1v1 duel. Then reconstruct the final result.

In [41]:
import pandas as pd
import numpy as np

teams = pd.read_csv('./data_prepared/teams.csv')

TRAIN_YEARS = list(range(1998, 2004))
TEST_YEARS  = list(range(2005, 2009))

df = teams.sort_values(['franchID', 'year']).copy()

train = df[df['year'].isin(TRAIN_YEARS)].copy()
test  = df[df['year'].isin(TEST_YEARS)].copy()

In [42]:
teams.head()

,year,lgID,tmID,franchID,confID,divID,rank,playoff,G,W,...,BenchMinor,PPG,PPC,SHA,PKG,PKC,SHF,coach_changed,conference_rank,prev_conference_rank
0,1993,NHL,ANA,ANA,WC,PC,0.500000,NaN,1.000000,0.452830,...,0.458333,0.329897,0.562334,0.571429,0.438776,0.596010,0.250000,0,0.571429,NaN
1,1994,NHL,ANA,ANA,WC,PC,0.833333,NaN,0.000000,0.132075,...,0.291667,0.010309,0.100796,0.333333,0.234694,0.109726,0.111111,0,0.785714,0.571429
2,1995,NHL,ANA,ANA,WC,PC,0.500000,NaN,0.944444,0.490566,...,0.416667,0.391753,0.694960,0.190476,0.581633,0.683292,0.277778,0,0.500000,0.785714
3,1996,NHL,ANA,ANA,WC,PC,0.166667,CSF,0.944444,0.509434,...,0.416667,0.350515,0.448276,0.476190,0.387755,0.466334,0.250000,0,0.214286,0.500000
4,1997,NHL,ANA,ANA,WC,PC,0.833333,NaN,0.944444,0.320755,...,0.166667,0.247423,0.604775,0.428571,0.489796,0.615960,0.250000,0,0.785714,0.214286


# Naive Model :

In [43]:
test['prev_conference_rank'] = test.groupby('tmID')['conference_rank'].shift(1)
test['prev_confID']    = test.groupby('tmID')['confID'].shift(1)

test.loc[test['confID'] != test['prev_confID'], 'prev_conference_rank'] = np.nan
test = test.drop(columns='prev_confID')

test['prev_conference_rank'] = test['prev_conference_rank'].astype('Float64')

In [44]:
valid = test.dropna(subset=['prev_conference_rank', 'conference_rank']).copy()
valid['err'] = valid['prev_conference_rank'] - valid['conference_rank']

mse  = (valid['err'] ** 2).mean()
rmse = np.sqrt(mse)
print(f"MSE  (normalized): {mse:.6f}")
print(f"RMSE (normalized): {rmse:.4f}")

valid['n_teams'] = valid.groupby(['year','confID'])['conference_rank'].transform('size')
valid['err_pos'] = valid['err'] * valid['n_teams']

mae_pos  = valid['err_pos'].abs().mean()
rmse_pos = np.sqrt((valid['err_pos'] ** 2).mean())
print(f"MAE  : {mae_pos:.2f}")
print(f"RMSE : {rmse_pos:.2f}")

MSE  (normalized): 0.099748
RMSE (normalized): 0.3158
MAE  : 3.65
RMSE : 4.71


The source data contains season-level aggregates, not individual game dates. The sequence below therefore uses the 20 previous seasons in which both teams have a record. The matchup feature is populated from the 12 most recent available head-to-head seasons.

Each timestep contains three values: Team A's overall win rate, Team B's overall win rate, and the normalized head-to-head score difference. Missing historical matchups are zero-filled; no information from the target season is used in the input.

In [ ]:
"""
Matchup model -> per-team wins -> conference points/ranking, plus a naive
baseline and comparison charts.

Key changes:
- Ties removed. Binary target (win vs non-win) is era-consistent: a
  "non-win" is L/T/OTL pre-2005 and L/OTL post-2005. Rows are iterated as
  dicts so the "T" column name never collides with pandas internals.
- Points recomputed as 2*expected_wins + alpha*expected_nonwins, where
  alpha = historical fraction of non-wins that earned 1 point (T+OTL).
- Naive baseline = carry-forward previous conference rank (re-ranked in
  the current conference), evaluated on exactly the same teams the model
  is evaluated on.
- Charts: metric bars, error histogram, predicted-vs-actual scatter,
  per-season MAE.
"""
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ config
SEQUENCE_LENGTH = 20
EPOCHS = 60
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

TRAIN_YEARS = list(range(1998, 2004))
TEST_YEARS  = list(range(2005, 2009))


# ------------------------------------------------------------------ io
def load_table(prepared_name, raw_name):
    for path in (Path("data_prepared") / prepared_name, Path("data") / raw_name):
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(f"Missing {prepared_name} and {raw_name}")


teams_history = load_table("teams.csv", "Teams.csv")
head_to_head  = pd.read_csv(Path("data") / "TeamVsTeam.csv")
actual_teams  = pd.read_csv(Path("data") / "Teams.csv")

teams_history = teams_history[teams_history["lgID"].eq("NHL")].copy()
head_to_head  = head_to_head[head_to_head["lgID"].eq("NHL")].copy()
actual_teams  = actual_teams[actual_teams["lgID"].eq("NHL")].copy()

teams_history["win_rate"] = (
    teams_history["W"].div(teams_history["G"].replace(0, np.nan)).fillna(0.0)
)

team_by_year = teams_history.set_index(["year", "tmID"])
h2h_by_year  = head_to_head.set_index(["year", "tmID", "oppID"])

team_records = {(int(y), t): r for (y, t), r in team_by_year.iterrows()}
h2h_records  = {(int(y), t, o): r for (y, t, o), r in h2h_by_year.iterrows()}


def safe_float(v):
    return float(v) if pd.notna(v) else 0.0


# ------------------------------------------------------------------ features
def sequence_for(team_a, team_b, target_year):
    """Up to SEQUENCE_LENGTH past seasons where both teams have a record."""
    rows = []
    for year in sorted(y for y in teams_history["year"].unique() if y < target_year):
        a = team_records.get((int(year), team_a))
        b = team_records.get((int(year), team_b))
        if a is None or b is None:
            continue

        record = h2h_records.get((int(year), team_a, team_b))
        h2h_score = 0.0
        if record is not None:
            wins_vs   = safe_float(record.get("W"))
            losses_vs = sum(safe_float(record.get(c)) for c in ("L", "T", "OTL"))
            total_vs  = wins_vs + losses_vs
            h2h_score = (wins_vs - losses_vs) / max(total_vs, 1.0)

        rows.append([
            float(a["win_rate"]),
            float(b["win_rate"]),
            h2h_score,
            float(a["conference_rank"]),
            float(b["conference_rank"]),
        ])

    rows = rows[-SEQUENCE_LENGTH:]
    padding = [[0.0] * 5] * (SEQUENCE_LENGTH - len(rows))
    return np.asarray(padding + rows, dtype=np.float32)


def make_examples(years):
    """Binary targets: [P(win), P(non-win)]. Non-win = L + T + OTL."""
    sequences, targets, metadata = [], [], []
    for row in head_to_head[head_to_head["year"].isin(years)].to_dict("records"):
        wins    = safe_float(row.get("W"))
        nonwins = sum(safe_float(row.get(c)) for c in ("L", "T", "OTL"))
        total   = wins + nonwins
        if total <= 0:
            continue
        sequences.append(sequence_for(row["tmID"], row["oppID"], int(row["year"])))
        targets.append([wins / total, nonwins / total])
        metadata.append((int(row["year"]), row["tmID"], row["oppID"], int(round(total))))
    if not sequences:
        raise ValueError(f"No examples for {list(years)}")
    return np.stack(sequences), np.asarray(targets, dtype=np.float32), metadata


X_train, y_train, _ = make_examples(TRAIN_YEARS)
X_test,  y_test,  test_metadata = make_examples(TEST_YEARS)

feature_mean = X_train.reshape(-1, 5).mean(axis=0)
feature_std  = X_train.reshape(-1, 5).std(axis=0)
feature_std[feature_std < 1e-8] = 1.0
X_train = (X_train - feature_mean) / feature_std
X_test  = (X_test  - feature_mean) / feature_std


# ------------------------------------------------------------------ model
class MatchupLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm   = nn.LSTM(input_size=5, hidden_size=32, batch_first=True)
        self.output = nn.Linear(32, 2)

    def forward(self, sequence):
        _, (hidden, _) = self.lstm(sequence)
        return self.output(hidden[-1])


model = MatchupLSTM().to(DEVICE)
train_sequences = torch.from_numpy(X_train).to(DEVICE)
train_targets   = torch.from_numpy(y_train).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for _ in range(EPOCHS):
    optimizer.zero_grad()
    logits = model(train_sequences)
    loss   = -(train_targets * torch.log_softmax(logits, dim=1)).sum(dim=1).mean()
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    probabilities = torch.softmax(
        model(torch.from_numpy(X_test).to(DEVICE)), dim=1
    ).cpu().numpy()

print(f"LSTM device: {DEVICE} | epochs: {EPOCHS} | test pairs: {len(test_metadata)}")


# ------------------------------------------------------------------ per-pair predictions
predictions = pd.DataFrame([
    {
        "year": year,
        "team_a": team_a,
        "team_b": team_b,
        "win_probability":  float(p[0]),
        "loss_probability": float(p[1]),
        "scheduled_games":  games,
    }
    for (year, team_a, team_b, games), p in zip(test_metadata, probabilities)
])

team_conferences = actual_teams[["year", "tmID", "confID"]].drop_duplicates(["year", "tmID"])
predictions = predictions.merge(
    team_conferences.rename(columns={"tmID": "team_a", "confID": "confID_a"}),
    on=["year", "team_a"], how="left",
).merge(
    team_conferences.rename(columns={"tmID": "team_b", "confID": "confID_b"}),
    on=["year", "team_b"], how="left",
)

# TeamVsTeam stores (A,B) and (B,A); keep one orientation per unordered pair.
ranking_predictions = predictions.assign(
    pair_key=predictions.apply(lambda r: tuple(sorted((r["team_a"], r["team_b"]))), axis=1)
).drop_duplicates(["year", "pair_key"])


# ------------------------------------------------------------------ team-level reconstruction
# Each team's expected wins = sum over matchups of scheduled_games * P(win vs opponent).
team_predictions = pd.concat([
    ranking_predictions.assign(
        tmID=ranking_predictions["team_a"],
        confID=ranking_predictions["confID_a"],
        expected_wins=ranking_predictions["scheduled_games"] * ranking_predictions["win_probability"],
    )[["year", "confID", "tmID", "scheduled_games", "expected_wins"]],
    ranking_predictions.assign(
        tmID=ranking_predictions["team_b"],
        confID=ranking_predictions["confID_b"],
        expected_wins=ranking_predictions["scheduled_games"] * ranking_predictions["loss_probability"],
    )[["year", "confID", "tmID", "scheduled_games", "expected_wins"]],
])

predicted_ranking = (
    team_predictions
    .groupby(["year", "confID", "tmID"], as_index=False)
    .agg(scheduled_games=("scheduled_games", "sum"),
         expected_wins=("expected_wins",  "sum"))
)


# alpha = historical fraction of non-wins that earned 1 point (T + OTL / (L+T+OTL))
def compute_alpha(years):
    num = 0.0
    den = 0.0
    for row in head_to_head[head_to_head["year"].isin(years)].to_dict("records"):
        l   = safe_float(row.get("L"))
        t   = safe_float(row.get("T"))
        otl = safe_float(row.get("OTL"))
        den += l + t + otl
        num += t + otl
    return num / max(den, 1.0)


ALPHA = compute_alpha(TRAIN_YEARS)
print(f"OTL+T share of non-wins (train): {ALPHA:.3f}")

predicted_ranking["expected_nonwins"] = (
    predicted_ranking["scheduled_games"] - predicted_ranking["expected_wins"]
)
# Points = 2*W + 1*(non-wins that earned a point)
predicted_ranking["expected_points"] = (
    2 * predicted_ranking["expected_wins"] + ALPHA * predicted_ranking["expected_nonwins"]
)
predicted_ranking["predicted_rank"] = (
    predicted_ranking.groupby(["year", "confID"])["expected_points"]
    .rank(method="first", ascending=False).astype(int)
)


# ------------------------------------------------------------------ actuals + naive baseline
prepared_teams = load_table("teams.csv", "Teams.csv")
prepared_teams = prepared_teams[prepared_teams["lgID"].eq("NHL")].copy()

# conference_rank is normalized to [0,1]; re-ranking recovers the ordinal rank.
prepared_teams["conference_rank_number"] = (
    prepared_teams.groupby(["year", "confID"])["conference_rank"]
    .rank(method="first", ascending=True).astype("Int64")
)
prepared_teams = prepared_teams.sort_values(["tmID", "year"]).reset_index(drop=True)
prepared_teams["previous_conference_rank_number"] = (
    prepared_teams.groupby("tmID")["conference_rank_number"].shift(1)
)

actual = prepared_teams[prepared_teams["year"].isin(TEST_YEARS)][
    ["year", "tmID", "confID", "conference_rank_number",
     "W", "L", "T", "OTL", "Pts"]
].rename(columns={
    "conference_rank_number": "target_rank",
    "W":   "target_wins",
    "L":   "target_losses",
    "T":   "target_ties",
    "OTL": "target_otl",
    "Pts": "target_points",
})

previous_ranks = prepared_teams[["year", "tmID", "previous_conference_rank_number"]].copy()
previous_ranks["year"] = previous_ranks["year"] + 1
previous_ranks = previous_ranks.rename(
    columns={"previous_conference_rank_number": "previous_rank"}
)

comparison = predicted_ranking.merge(
    actual, on=["year", "tmID", "confID"], how="inner"
).merge(
    previous_ranks[["year", "tmID", "previous_rank"]],
    on=["year", "tmID"], how="left",
)

# --- Reconstruct actual points in their natural unit (not the normalized one) ---
raw_teams = pd.read_csv(Path("data") / "Teams.csv")
raw_teams = raw_teams[raw_teams["lgID"].eq("NHL")].copy()
raw_teams["real_points"] = (
    2 * raw_teams["W"].fillna(0)
    + raw_teams["T"].fillna(0)
    + raw_teams["OTL"].fillna(0)
)
real_points_lookup = raw_teams[["year", "tmID", "real_points"]].rename(
    columns={"real_points": "target_points_real"}
)

valid = valid.merge(real_points_lookup, on=["year", "tmID"], how="left")

# Naive: predict this year's rank by re-ranking the carried-forward rank.
naive = comparison[["year", "tmID", "confID", "previous_rank"]].copy()
naive["naive_rank"] = (
    naive.groupby(["year", "confID"])["previous_rank"]
    .rank(method="first", ascending=True)
)
comparison = comparison.merge(
    naive[["year", "tmID", "naive_rank"]], on=["year", "tmID"], how="left"
)

comparison["model_err"] = comparison["predicted_rank"] - comparison["target_rank"]
comparison["naive_err"] = comparison["naive_rank"]     - comparison["target_rank"]

valid = comparison.dropna(
    subset=["predicted_rank", "naive_rank", "target_rank"]
).copy()

model_mae   = valid["model_err"].abs().mean()
naive_mae   = valid["naive_err"].abs().mean()
model_rmse  = float(np.sqrt((valid["model_err"] ** 2).mean()))
naive_rmse  = float(np.sqrt((valid["naive_err"] ** 2).mean()))
model_spear = valid[["predicted_rank", "target_rank"]].corr(method="spearman").iloc[0, 1]
naive_spear = valid[["naive_rank",     "target_rank"]].corr(method="spearman").iloc[0, 1]

print(f"\nEvaluated teams: {len(valid)}")
print(f"Model  -> MAE {model_mae:.3f} | RMSE {model_rmse:.3f} | Spearman {model_spear:.3f}")
print(f"Naive  -> MAE {naive_mae:.3f} | RMSE {naive_rmse:.3f} | Spearman {naive_spear:.3f}")

# ------------------------------------------------------------------ per-season / per-conference table
show_cols = [
    "confID", "tmID", "previous_rank", "predicted_rank", "naive_rank",
    "expected_points", "target_rank", "target_points",
]
for year in TEST_YEARS:
    for conf in sorted(valid.loc[valid["year"].eq(year), "confID"].dropna().unique()):
        print(f"\nSeason {year} - conference {conf}")
        print(
            valid.loc[
                valid["year"].eq(year) & valid["confID"].eq(conf), show_cols
            ].sort_values("target_rank").to_string(index=False)
        )

# ------------------------------------------------------------------ charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Metric bars
ax = axes[0, 0]
metrics = ["MAE", "RMSE"]
x = np.arange(len(metrics))
w = 0.35
ax.bar(x - w / 2, [model_mae, model_rmse], w, label="Model", color="#4C72B0")
ax.bar(x + w / 2, [naive_mae, naive_rmse], w, label="Naive", color="#DD8452")
for i, (m, n) in enumerate(zip([model_mae, model_rmse], [naive_mae, naive_rmse])):
    ax.text(i - w / 2, m + 0.03, f"{m:.2f}", ha="center", fontsize=9)
    ax.text(i + w / 2, n + 0.03, f"{n:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("Conference rank error (positions)")
ax.set_title("Overall rank-prediction error")
ax.legend(); ax.grid(True, alpha=0.3)

# 2) Error histogram
ax = axes[0, 1]
bins = np.arange(-10.5, 11.5, 1)
ax.hist(valid["model_err"], bins=bins, alpha=0.55, label="Model", color="#4C72B0")
ax.hist(valid["naive_err"], bins=bins, alpha=0.55, label="Naive", color="#DD8452")
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Predicted rank − actual rank")
ax.set_ylabel("Teams")
ax.set_title("Rank error distribution")
ax.legend(); ax.grid(True, alpha=0.3)

# 3) Predicted vs actual scatter
ax = axes[1, 0]
ax.scatter(valid["target_rank"], valid["predicted_rank"],
           s=22, alpha=0.5, label="Model", color="#4C72B0")
ax.scatter(valid["target_rank"], valid["naive_rank"],
           s=22, alpha=0.5, label="Naive", color="#DD8452")
lims = [0.5, valid["target_rank"].max() + 0.5]
ax.plot(lims, lims, "k--", lw=0.9, alpha=0.6)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual rank"); ax.set_ylabel("Predicted rank")
ax.set_title("Predicted vs actual rank")
ax.legend(); ax.grid(True, alpha=0.3)

# 4) Per-season MAE
ax = axes[1, 1]
per_season = valid.groupby("year").agg(
    model_mae=("model_err", lambda s: s.abs().mean()),
    naive_mae=("naive_err", lambda s: s.abs().mean()),
)
ax.plot(per_season.index, per_season["model_mae"], marker="o", label="Model", color="#4C72B0")
ax.plot(per_season.index, per_season["naive_mae"], marker="o", label="Naive", color="#DD8452")
ax.set_xticks(per_season.index)
ax.set_xlabel("Season"); ax.set_ylabel("MAE (rank positions)")
ax.set_title("Per-season rank MAE")
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle("Conference-rank prediction: model vs naive carry-forward", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig("ranking_comparison.png", dpi=130)
print("\nSaved chart -> ranking_comparison.png")


LSTM device: cuda | epochs: 60 | test pairs: 3030
OTL+T share of non-wins (train): 0.312

Evaluated teams: 88
Model  -> MAE 3.943 | RMSE 4.987 | Spearman 0.332
Naive  -> MAE 4.170 | RMSE 5.303 | Spearman 0.231

Season 2006 - conference EC
confID tmID  previous_rank  predicted_rank  naive_rank  expected_points  target_rank  target_points
    EC  BUF              9              10         9.0        91.873211            1       0.833333
    EC  NJD              6               5         6.0       103.173652            2       0.777778
    EC  OTT              4               1         4.0       111.498133            3       0.759259
    EC  PIT             15              12        15.0        85.467696            4       0.759259
    EC  ATL             10              15        10.0        81.589972            5       0.685185
    EC  NYR             13               6        13.0        94.684791            6       0.657407
    EC  TBL              1               3         1.0       